In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType, IntegerType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *
import time


spark = (
    SparkSession.builder
    .appName("Pre-partitioning")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 01:51:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 01:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# de-active broadcast joins since we are running joins on small datasets but do not want broadcast for demo
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# dont touch this

# addColumns(initialTable, 3) => dataframe wit columns "id", "newCol1", "newCol2", "newCol3"
def addColumns(df, n):
    new_columns = [f"id * {i} as newCol{i}" for i in range(1, n + 1)]
    return df.selectExpr("id", *new_columns)


initialTable = spark.range(1, 7000000).repartition(10)

narrowTable = spark.range(1, 3000000).repartition(7)

In [ ]:
# scenario 1
start = time.time()
wideTable = addColumns(initialTable, 30)

join1 = wideTable.join(narrowTable, "id")


join1.explain()
join1.count()
end = time.time()

print("Took", end - start, "seconds")

In [ ]:
# scenario 2
start = time.time()
# will use a hash parititoner so rows with same value for that id will sit in same partition
altNarrow = narrowTable.repartition("id") 
altInitial = initialTable.repartition("id")

join2 = altInitial.join(altNarrow, "id")

result2 = addColumns(join2, 30)

result2.explain()
result2.count()
end = time.time()
print("Took", end - start, "seconds")

In [ ]:
# why is it so much faster?
# first reason is that it does not do the random round robin repartiion because it notices we will repartiion by id right after.
# second reson - is that the same ids from both dataframes will be co-partitioned already so less shuffles 
#
# LESSON - partition early, Partitoing late is AT BEST what spark naturally does. See Useless comment in cells below.

In [ ]:

# Scenario 3 
start = time.time()

enhanceColumnsFirst = addColumns(initialTable, 30)
repartionedNarrow = narrowTable.repartition("id")
repartionEnhanced = enhanceColumnsFirst.repartition("id") # USELESS !

result3 = repartionEnhanced.join(repartionedNarrow, "id")

result3.explain()
result3.count()
end = time.time()
print(f"Took {end - start} seconds")

In [ ]:

# Notice that scenario 3 is slower than 2 thats because in 2 we add columns after join
# Less data you have before the join less data you need to shuffle around. 
# Also we did force the round robin repartion by including the add columns between reparition and hash parition

In [ ]:

# Basically identical to lesson 1 since even though both have same parition amount ids are not guaranteed to be in same partiton
# so several shuffles occur
initialTable.join(narrowTable.repartition(10), "id").explain()